In [1]:
!pip install fairlearn pgmpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.0/240.0 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [2]:
import random
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination

# ---------------- reproducibility ----------------
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

In [3]:
def preprocess_compas_for_fair_bbn(csv_path='/kaggle/input/compas/compas-scores-two-years.csv', seed=42):
        
    # Load and filter COMPAS data
    df = pd.read_csv(csv_path)
    df = df.loc[
    (df['days_b_screening_arrest'] <= 30) &
    (df['days_b_screening_arrest'] >= -30) &
    (df['is_recid'] != -1) &
    (df['c_charge_degree'] != 'O') &
    (df['score_text'] != 'N/A'),
    ['age','c_charge_degree','race','age_cat','score_text','sex',
    'priors_count','days_b_screening_arrest','decile_score',
    'juv_other_count','juv_misd_count','juv_fel_count',
    'c_charge_desc','is_recid','two_year_recid','c_jail_in','c_jail_out']
    ].reset_index(drop=True)
    
    
    # Sensitive label mapping
    race_map = {"African-American":0,"Caucasian":1,"Hispanic":2,"Other":3,"Asian":4,"Native American":5}
    sex_map = {"Male":0,"Female":1}
    df['race_label'] = df['race'].map(race_map)
    df['sex_label'] = df['sex'].map(sex_map)
    
    # Jail time calculation
    df['c_jail_in'] = pd.to_datetime(df['c_jail_in'])
    df['c_jail_out'] = pd.to_datetime(df['c_jail_out'])
    df['jail_time'] = (df['c_jail_out'] - df['c_jail_in']).dt.days
    df['jail_time'] = df['jail_time'].fillna(0)
    df = df.drop(columns=['c_jail_in','c_jail_out'])
    
    # Target variable
    y = df['two_year_recid'].values
    sens_race = df['race_label']
    sens_sex = df['sex_label']
    
    # Columns
    numeric_cols = ['age','priors_count','days_b_screening_arrest','decile_score',
                    'jail_time','juv_other_count','juv_misd_count','juv_fel_count']
    categorical_cols = ['c_charge_degree','race','age_cat','score_text','sex','c_charge_desc']
    
    # Preprocessor for NN (scaled + one-hot)
    preproc = ColumnTransformer([
        ('num', Pipeline([('scaler', StandardScaler())]), numeric_cols),
        ('cat', Pipeline([('ohe', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols)
    ])
    
    # BBN preprocessing: discretize numerics, encode categoricals
    bbn_df = df.copy()
    for col in numeric_cols:
        bbn_df[col] = pd.qcut(bbn_df[col], 5, labels=False, duplicates='drop').astype(int)
    for col in categorical_cols + ['race','sex']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)
    
    # Split train/test
    X = df.drop(columns=['is_recid','two_year_recid','race_label','sex_label'])
    X_train_raw, X_test_raw, y_train, y_test, sens_race_train, sens_race_test, sens_sex_train, sens_sex_test = \
        train_test_split(X, y, sens_race, sens_sex, test_size=0.2, stratify=y, random_state=seed)
    
    # NN preprocessing
    X_train_nn = preproc.fit_transform(X_train_raw)
    X_test_nn = preproc.transform(X_test_raw)
    
    # BBN aligned splits
    bbn_train = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test = bbn_df.loc[X_test_raw.index].reset_index(drop=True)
    
    return {
        'X_train_nn': X_train_nn, 'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train, 'bbn_test_df': bbn_test,
        'y_train': y_train, 'y_test': y_test,
        'sens_race_train': sens_race_train.reset_index(drop=True),
        'sens_race_test': sens_race_test.reset_index(drop=True),
        'sens_sex_train': sens_sex_train.reset_index(drop=True),
        'sens_sex_test': sens_sex_test.reset_index(drop=True)
    }
    


In [4]:
# ------------------------ BBN ------------------------
class SimpleFairBBN:
    #list of features to include in the bbn and the target attribute
    def __init__(self, feature_names, target_name='target'):
        self.feature_names = feature_names
        self.target_name = target_name
        self.model = None #will hold DiscreteBayesianNetwork
        self.inference = None #will hold VariableElimination object for querying probabs
    #df_discrete is the dataset with discretised features and y is target labels.
    def fit(self, df_discrete, y, include_sensitive=True):
        #combine feature and target for BBN 
        data = df_discrete.copy().reset_index(drop=True)
        data[self.target_name] = y
        # Use top 6 features + optionally sensitive attrs
        candidate_features = list(df_discrete.columns)
        selected = candidate_features[:6]
        if include_sensitive: #this is optional, can mess around with this
            if 'race' in df_discrete.columns: selected.append('race')
            if 'sex' in df_discrete.columns: selected.append('sex')
        edges = [(f, self.target_name) for f in selected] #feature to target edge
        #fitting the BBN
        self.feature_names = selected
        self.model = DiscreteBayesianNetwork(edges)
        # BDeu estimator with small equivalent sample size for fairness smoothing
        self.model.fit(data, estimator=BayesianEstimator, prior_type='BDeu', equivalent_sample_size=5)
        self.inference = VariableElimination(self.model)

    def predict_proba(self, df_discrete):
        Xdf = df_discrete.reset_index(drop=True)
        probs = [] #we predict probability of where income = 1
        for _, row in Xdf.iterrows():
            evidence = {} #features that will be given to the BBN as observations
            used = 0 #this limits the no of features used as evidence 
            for f in self.feature_names:
                if f in row and not pd.isna(row[f]) and used<3: #make the feature isnt nan and used less than 3 times
                    try: evidence[f] = int(row[f]); used+=1 #convert feature val to int
                    except: continue
            try:
                if evidence: #query for inference if evidence exists
                    q = self.inference.query(variables=[self.target_name], evidence=evidence)
                else: #else marginal probability
                    q = self.inference.query(variables=[self.target_name])
                p1 = q.values[1] if len(q.values)==2 else 0.5 #q.values[1] gives probab of 1 
                #default to 0.5 if something went wrong and the values are not binary
            except: p1=0.5
            probs.append(p1) #p1 is the predicted probab of 1 row
        return np.array(probs)


In [5]:
# ---------------- Adapter + Adversary ----------------
#actual predictor which ips the vals of the bbn
class AdapterNN(nn.Module):
    def __init__(self, in_dim=3, hidden_dim=32):  # 3 BBN marginals as input - [P(y|all), P(y|race), P(y|sex)]
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, hidden_dim//2)
        self.out = nn.Linear(hidden_dim//2,1) # 1 because binary classification 
    def forward(self, x, return_features=False):
        h = self.act(self.fc1(x))
        h2 = self.act(self.fc2(h))
        logit = self.out(h2)
        if return_features: 
            return logit,h2 #logit is wht we get raw before applying sigmoid 
        return logit
#mitigation and predicts sensitive attr from the adapters rep 
class AdversaryNN(nn.Module):
    def __init__(self, in_dim, race_classes, sex_classes): #ip is hidden dim of the adapter
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(in_dim,32), nn.ReLU())
        self.race_head = nn.Linear(32, race_classes) #diff predictors for race and sex
        self.sex_head = nn.Linear(32, sex_classes)
    def forward(self,x):
        s=self.shared(x)
        return self.race_head(s), self.sex_head(s) #returns logits for sex and race 


In [6]:
def train_fair_bbn_adapter_compas(data_dict, lambda_adv=0.5, alpha_bbn=0.5, epochs=60, batch_size=256, lr=1e-3, seed=42):
# Extract datasets from COMPAS preprocessing
    X_train_nn = data_dict['X_train_nn']; X_test_nn = data_dict['X_test_nn']
    bbn_train_df = data_dict['bbn_train_df']; bbn_test_df = data_dict['bbn_test_df']
    y_train = data_dict['y_train']; y_test = data_dict['y_test']
    sens_race_train = data_dict['sens_race_train']; sens_race_test = data_dict['sens_race_test']
    sens_sex_train = data_dict['sens_sex_train']; sens_sex_test = data_dict['sens_sex_test']


# ---------------- Baseline NN ----------------
    print("Training baseline MLP...")
    baseline = MLPClassifier(hidden_layer_sizes=(64,32), max_iter=300, random_state=seed)
    baseline.fit(X_train_nn, y_train)
    base_pred = baseline.predict(X_test_nn)
    base_acc = accuracy_score(y_test, base_pred)
    base_race_dp = demographic_parity_difference(y_test, base_pred, sensitive_features=sens_race_test)
    base_race_eo = equalized_odds_difference(y_test, base_pred, sensitive_features=sens_race_test)
    base_sex_dp = demographic_parity_difference(y_test, base_pred, sensitive_features=sens_sex_test)
    base_sex_eo = equalized_odds_difference(y_test, base_pred, sensitive_features=sens_sex_test)
    print(f"Baseline MLP Accuracy: {base_acc:.4f}")

# ---------------- Fair BBN ----------------
    print("Training fair BBN...")
    bbn = SimpleFairBBN(feature_names=list(bbn_train_df.columns))
    bbn.fit(bbn_train_df, y_train, include_sensitive=True)
    
    # Generate marginals for adapter input
    p_all = bbn.predict_proba(bbn_train_df).reshape(-1,1)
    p_race = bbn.predict_proba(bbn_train_df[['race']]).reshape(-1,1)
    p_sex = bbn.predict_proba(bbn_train_df[['sex']]).reshape(-1,1)
    adapter_in_train = torch.tensor(np.hstack([p_all,p_race,p_sex]), dtype=torch.float32)
    
    p_all_test = bbn.predict_proba(bbn_test_df).reshape(-1,1)
    p_race_test = bbn.predict_proba(bbn_test_df[['race']]).reshape(-1,1)
    p_sex_test = bbn.predict_proba(bbn_test_df[['sex']]).reshape(-1,1)
    adapter_in_test = torch.tensor(np.hstack([p_all_test,p_race_test,p_sex_test]), dtype=torch.float32)

# Convert labels & sensitive attrs
    y_train_t = torch.tensor(y_train.reshape(-1,1), dtype=torch.float32)
    y_test_t  = torch.tensor(y_test.reshape(-1,1), dtype=torch.float32)
    race_train_t = torch.tensor(sens_race_train.values.astype(int), dtype=torch.long)
    sex_train_t  = torch.tensor(sens_sex_train.values.astype(int), dtype=torch.long)

    train_loader = DataLoader(
        TensorDataset(adapter_in_train, y_train_t, race_train_t, sex_train_t),
        batch_size=batch_size, shuffle=True
    )

# Initialize adapter + adversary
    adapter = AdapterNN(in_dim=3, hidden_dim=64)
    adversary = AdversaryNN(in_dim=32,
                            race_classes=len(sens_race_train.unique()),
                            sex_classes=len(sens_sex_train.unique()))
    adapter_opt = optim.Adam(adapter.parameters(), lr=lr)
    adv_opt = optim.Adam(adversary.parameters(), lr=lr)
    pred_loss_fn = nn.BCEWithLogitsLoss()
    adv_loss_fn = nn.CrossEntropyLoss()

# ---------------- Training loop ----------------
    print("Training adapter with adversarial + BBN fairness...")
    for epoch in range(epochs):
        adapter.train(); adversary.train()
        total_adapter_loss=0.0; total_adv_loss=0.0
        for batch_in, batch_y, batch_race, batch_sex in train_loader:
            # --- Train adversary ---
            with torch.no_grad():
                _, features = adapter(batch_in, return_features=True)
            adv_opt.zero_grad()
            r_logits, s_logits = adversary(features)
            adv_loss = (adv_loss_fn(r_logits,batch_race) + adv_loss_fn(s_logits,batch_sex))/2.0
            adv_loss.backward(); adv_opt.step()
            total_adv_loss += adv_loss.item()
    
            # --- Train adapter ---
            adapter_opt.zero_grad()
            logit, features = adapter(batch_in, return_features=True)
            pred_loss = pred_loss_fn(logit,batch_y)
            r_logits2, s_logits2 = adversary(features)
            adv_loss_for_adapter = (adv_loss_fn(r_logits2,batch_race) + adv_loss_fn(s_logits2,batch_sex))/2.0
            # fairness penalty across race groups
            dp_penalty = torch.abs(features[batch_race==0].mean() - features[batch_race==1].mean())
            total_loss = pred_loss - lambda_adv*adv_loss_for_adapter + alpha_bbn*dp_penalty
            total_loss.backward()
            adapter_opt.step()
            total_adapter_loss += total_loss.item()
    
        if epoch % 10==0 or epoch==epochs-1:
            print(f"Epoch {epoch:3d} | Adv Loss: {total_adv_loss/len(train_loader):.4f} | Adapter Loss: {total_adapter_loss/len(train_loader):.4f}")
    
    # ---------------- Evaluation ----------------
    adapter.eval()
    with torch.no_grad():
        test_logit,_ = adapter(adapter_in_test, return_features=True)
        test_probs = torch.sigmoid(test_logit.cpu()).numpy().flatten()
        test_pred = (test_probs>0.5).astype(int)

    adv_acc = accuracy_score(y_test, test_pred)
    adv_race_dp = demographic_parity_difference(y_test, test_pred, sensitive_features=sens_race_test)
    adv_race_eo = equalized_odds_difference(y_test, test_pred, sensitive_features=sens_race_test)
    adv_sex_dp = demographic_parity_difference(y_test, test_pred, sensitive_features=sens_sex_test)
    adv_sex_eo = equalized_odds_difference(y_test, test_pred, sensitive_features=sens_sex_test)

# Print results
    print("\nBASELINE MLP RESULTS:")
    print(f" Accuracy: {base_acc:.4f}")
    print(f" Race DP: {base_race_dp:.4f}, Race EO: {base_race_eo:.4f}")
    print(f" Sex  DP: {base_sex_dp:.4f}, Sex  EO: {base_sex_eo:.4f}")
    
    print("\nBBN + Adapter (Adversarial + Fairness) RESULTS:")
    print(f" Accuracy: {adv_acc:.4f}")
    print(f" Race DP: {adv_race_dp:.4f}, Race EO: {adv_race_eo:.4f}")
    print(f" Sex  DP: {adv_sex_dp:.4f}, Sex  EO: {adv_sex_eo:.4f}")
    
    return {
        'baseline':{
            'pred':base_pred,'acc':base_acc,
            'race_dp':base_race_dp,'race_eo':base_race_eo,
            'sex_dp':base_sex_dp,'sex_eo':base_sex_eo
        },
        'bbn_adapter':{
            'pred':test_pred,'acc':adv_acc,
            'race_dp':adv_race_dp,'race_eo':adv_race_eo,
            'sex_dp':adv_sex_dp,'sex_eo':adv_sex_eo
        }
    }



In [7]:
# ---------------- Run ----------------
if __name__=='__main__':
    print("Preprocessing dataset...")
    data_dict = preprocess_compas_for_fair_bbn()
    results = train_fair_bbn_adapter_compas(data_dict, lambda_adv=0.5, alpha_bbn=0.5, epochs=60)

Preprocessing dataset...
Training baseline MLP...
Baseline MLP Accuracy: 0.6372
Training fair BBN...
Training adapter with adversarial + BBN fairness...
Epoch   0 | Adv Loss: 1.2001 | Adapter Loss: 0.0922
Epoch  10 | Adv Loss: 0.8432 | Adapter Loss: 0.2684
Epoch  20 | Adv Loss: 0.8076 | Adapter Loss: 0.2844
Epoch  30 | Adv Loss: 0.8029 | Adapter Loss: 0.2880
Epoch  40 | Adv Loss: 0.8033 | Adapter Loss: 0.2877
Epoch  50 | Adv Loss: 0.8024 | Adapter Loss: 0.2883
Epoch  59 | Adv Loss: 0.8025 | Adapter Loss: 0.2883

BASELINE MLP RESULTS:
 Accuracy: 0.6372
 Race DP: 0.5538, Race EO: 0.6558
 Sex  DP: 0.1319, Sex  EO: 0.1136

BBN + Adapter (Adversarial + Fairness) RESULTS:
 Accuracy: 0.5449
 Race DP: 0.0000, Race EO: 0.0000
 Sex  DP: 0.0000, Sex  EO: 0.0000
